In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, hour, dayofweek
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


In [2]:
spark = SparkSession.builder \
    .appName("BigQuery_to_PySpark_ML") \
    .getOrCreate()

26/02/28 07:20:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# Configuración de nombres (Ajusta el PROJECT_ID con el tuyo de Google Cloud)
PROJECT_ID = "big-data-lunes-2026-1" 
DATASET_ID = "up"
TABLE_ID = "bank_transactions"
BUCKET_TEMP = "big-data-lunes-20260223" # Requerido para escribir en BigQuery

In [4]:
df_bank = spark.read.format("bigquery") \
    .option("table", f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}") \
    .load()

In [5]:
df_bank.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- from_user_id: string (nullable = true)
 |-- from_name: string (nullable = true)
 |-- from_country: string (nullable = true)
 |-- from_bank: string (nullable = true)
 |-- from_location: string (nullable = true)
 |-- to_user_id: string (nullable = true)
 |-- to_name: string (nullable = true)
 |-- to_country: string (nullable = true)
 |-- to_bank: string (nullable = true)
 |-- to_location: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- is_suspicious: boolean (nullable = true)
 |-- suspicious_pattern: string (nullable = true)



---
# BLOQUE A: MODELO DE CLUSTERING (Segmentación de Riesgo)
---

In [6]:
df_base = df_bank.select(
    "amount", "from_country", "from_bank", "transaction_type", "timestamp", "is_suspicious"
).withColumn("hour", hour(col("timestamp"))) \
 .withColumn("day", dayofweek(col("timestamp"))) \
 .withColumn("label_real", col("is_suspicious").cast("double")) \
 .na.drop()

In [7]:
cat_cols = ["from_country", "from_bank", "transaction_type"]
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="skip") for c in cat_cols]
encoder = OneHotEncoder(inputCols=[c+"_idx" for c in cat_cols], outputCols=[c+"_vec" for c in cat_cols])

assembler = VectorAssembler(
    inputCols=["amount", "hour", "day"] + [c+"_vec" for c in cat_cols],
    outputCol="features_raw"
)

In [8]:
%%time
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled")

prep_pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler])
prep_model = prep_pipeline.fit(df_base)
df_final = prep_model.transform(df_base)

CPU times: user 58.6 ms, sys: 5.09 ms, total: 63.7 ms
Wall time: 35.4 s


In [10]:
%%time
print("Calculando el método del codo para K de 2 a 15...")
cost = []
n = 6
for k in range(2, n):
    kmeans_test = KMeans(featuresCol="features_scaled", k=k, seed=42)
    model_test = kmeans_test.fit(df_final)
    # En versiones recientes de Spark usamos getSummary().trainingCost
    cost.append(model_test.summary.trainingCost)

Calculando el método del codo para K de 2 a 15...


CPU times: user 53.3 ms, sys: 15.1 ms, total: 68.4 ms
Wall time: 1min 16s


In [11]:
cost

[13370375.519681444,
 12849634.991010014,
 12466235.261444682,
 11887925.054516617]

In [13]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import plotly.express as px
import pandas as pd

# Crear DataFrame para el codo
df_elbow = pd.DataFrame({
    'K': list(range(2, n)),
    'WSS': cost
})

# Graficar
fig_elbow = px.line(
    df_elbow, x='K', y='WSS', 
    title='Método del Codo (Selección de K)',
    markers=True,
    template='plotly_white'
)

# Si .show() solo no funciona, forzamos el renderizador aquí
fig_elbow.show(renderer="notebook_connected")

In [14]:
# 4. ENTRENAMIENTO DEL MODELO K-MEANS FINAL Y PCA
print("\nEntrenando modelo K-Means final y aplicando PCA...")
k_optimo = 4 # Asume este K basado en el gráfico del codo anterior
kmeans_final = KMeans(featuresCol="features_scaled", k=k_optimo, predictionCol="cluster", seed=42)
model_final = kmeans_final.fit(df_final)
df_res = model_final.transform(df_final)

# Reducción de dimensionalidad con PCA para visualización 2D
pca = PCA(k=2, inputCol="features_scaled", outputCol="pca_features")
pca_model = pca.fit(df_res)
df_vis = pca_model.transform(df_res)


Entrenando modelo K-Means final y aplicando PCA...


In [15]:
# --- CONFIGURACIÓN DE SEGURIDAD ---
MAX_RECORDS_DRIVER = 15000  # Límite seguro para no saturar la RAM del Driver
print("Iniciando proceso de visualización segura...")

# 1. Calcular el conteo total de registros disponibles tras PCA
total_count = df_vis.count()
print(f"Total de registros en df_vis: {total_count}")

# 2. Determinar la fracción de muestreo óptima
# Si el total es menor al límite, usamos 1.0 (100%). Si es mayor, calculamos la fracción.
sampling_fraction = min(1.0, MAX_RECORDS_DRIVER / total_count)

if sampling_fraction < 1.0:
    print(f"El dataset es grande. Muestreando al {sampling_fraction*100:.2f}% para seguridad.")
else:
    print("Dataset manejable. Procediendo con todos los datos seleccionados.")

# 3. Ejecutar el SELECT y el muestreo antes de toPandas()
# Importante: El sample() ocurre en los workers, toPandas() solo trae el resultado final.
df_to_convert = df_vis.select(
    "pca_features", "cluster", "label_real", "amount", "from_bank"
).sample(withReplacement=False, fraction=sampling_fraction, seed=42)

# Ahora es seguro convertir
sample_pd = df_to_convert.toPandas()

# 4. PROCESAMIENTO EN PANDAS
# Expandir vector PCA a columnas x, y
coords = pd.DataFrame(
    sample_pd['pca_features'].apply(lambda x: x.toArray()).tolist(), 
    columns=['x', 'y']
)
sample_pd = pd.concat([sample_pd, coords], axis=1)

# Crear etiqueta legible para el gráfico
sample_pd['Tipo'] = sample_pd['label_real'].apply(lambda x: 'ALERTA: FRAUDE' if x == 1.0 else 'Normal')

# 5. GRÁFICO SCATTER (Plotly)
fig_clusters = px.scatter(
    sample_pd, x='x', y='y',
    color='cluster',
    symbol='Tipo',
    hover_data=['amount', 'from_bank'],
    title=f"Clusters de Transacciones y Fraude (K={k_optimo}) - Muestra Segura",
    template='plotly_dark',
    color_continuous_scale=px.colors.qualitative.Safe
)

# Ajuste visual para los sospechosos (X rojas)
fig_clusters.update_traces(marker=dict(size=10), selector=dict(marker_symbol='x'))

# Renderizado explícito para asegurar visualización en Notebooks
fig_clusters.show(renderer="notebook_connected")

Iniciando proceso de visualización segura...


Total de registros en df_vis: 460311
⚠️ El dataset es grande. Muestreando al 3.26% para seguridad.


Puntos clave para la clase:
1. El Bottleneck: Explica que la capa central (oculta) tiene menos neuronas que la entrada. Esto obliga a la red a quedarse solo con lo más importante de la transacción.
2. Umbral de Decisión (Threshold): En un Autoencoder real, definiríamos un valor $E$ (error). Si $Error > E$, disparamos una alerta de fraude.
3. Diferencia con Random Forest: El Random Forest necesita que le digas qué es fraude. El Autoencoder solo necesita saber qué es lo normal, lo cual es muy útil porque los defraudadores siempre cambian sus tácticas.

---
# BLOQUE B: MODELO DE CLASIFICACIÓN (Predicción de Fraude)
---

In [16]:
%%time 

df_fixed = df_bank.withColumn("label", col("is_suspicious").cast("double")) \
                  .withColumn("is_suspicious_num", col("is_suspicious").cast("double")) \
                  .na.drop(subset=["amount", "is_suspicious"])

idx_bank = StringIndexer(inputCol="from_bank", outputCol="bank_idx", handleInvalid="skip")
idx_type = StringIndexer(inputCol="transaction_type", outputCol="type_idx", handleInvalid="skip")

# Etapa 2: Encoder (Números -> Vectores Binarios)
ohe = OneHotEncoder(
    inputCols=["bank_idx", "type_idx"], 
    outputCols=["bank_vec", "type_vec"]
)

# Etapa 3: Ensamblar (Monto + Vectores OHE)
ass_c = VectorAssembler(
    inputCols=["amount", "bank_vec", "type_vec"], 
    outputCol="features_c"
)

# Etapa 4: Algoritmo
rf = RandomForestClassifier(labelCol="label", featuresCol="features_c", numTrees=50)

# Pipeline de Clasificación
pipe_rf = Pipeline(stages=[idx_bank, idx_type, ohe, ass_c, rf])

# Entrenamiento y Evaluación
train, test = df_fixed.randomSplit([0.8, 0.2], seed=42)
model_rf = pipe_rf.fit(train)
preds = model_rf.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(preds)
print(f"\n>>> AUC Final con OHE: {auc:.4f} <<<")


>>> AUC Final con OHE: 0.5000 <<<
CPU times: user 86.7 ms, sys: 19.2 ms, total: 106 ms
Wall time: 1min 12s


# Bloque C: Un autoencoder

In [17]:
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.functions import vector_to_array
import plotly.express as px
import sys

In [18]:
# 1. DEFINICIÓN DEL AUTOENCODER (MLP como aproximador)
# Capas: [Entrada, 8 (Bottleneck), 4, 2 (Salida Binaria)]
input_dim = len(df_final.select("features_scaled").first()[0])
layers = [input_dim, 8, 4, 2]

In [19]:
autoencoder = MultilayerPerceptronClassifier(
    layers=layers, seed=42, featuresCol="features_scaled", labelCol="label_real"
)

In [20]:
%%time
# Entrenamos (usualmente con una fracción de datos normales)
model_ae = autoencoder.fit(df_final)
df_predictions = model_ae.transform(df_final)

# Calculamos el Score de Anomalía (1 - probabilidad de ser 'Normal')
df_predictions = df_predictions.withColumn("p_array", vector_to_array("probability")) \
                               .withColumn("anomaly_score", 1 - col("p_array")[0])

CPU times: user 24.6 ms, sys: 4.47 ms, total: 29 ms
Wall time: 56.6 s


In [21]:
MAX_ROWS = 10000
total_rows = df_predictions.count()

print(f"Filas totales en el DataFrame: {total_rows}")

if total_rows > MAX_ROWS:
    print(f"¡Peligro! El dataset es demasiado grande ({total_rows} filas).")
    print(f"Tomando una muestra aleatoria para proteger la memoria del Driver...")
    # Calculamos la fracción necesaria para llegar al límite seguro
    fraction = MAX_ROWS / total_rows
    df_to_plot = df_predictions.sample(withReplacement=False, fraction=fraction, seed=42)
else:
    print("El dataset es de tamaño seguro. Procediendo sin muestreo.")
    df_to_plot = df_predictions

Filas totales en el DataFrame: 460311
¡Peligro! El dataset es demasiado grande (460311 filas).
Tomando una muestra aleatoria para proteger la memoria del Driver...


In [25]:
import plotly.graph_objects as go

# 1. Separamos los datos en el DataFrame de Pandas para aplicar estilos distintos
normales = sample_pd[sample_pd['label_real'] == 0]
fraudulentos = sample_pd[sample_pd['label_real'] == 1]

# 2. Creamos la figura usando Graph Objects para control total
fig = go.Figure()

# CAPA 1: Transacciones Normales (Gris y Traslúcidas)
fig.add_trace(go.Scatter(
    x=normales['amount'],
    y=normales['anomaly_score'],
    mode='markers',
    name='Normal (Bajo Riesgo)',
    marker=dict(
        color='lightgrey',
        size=6,
        opacity=0.3  # Traslúcido
    )
))

# CAPA 2: Transacciones Fraudulentas (Grandes y Rojas)
fig.add_trace(go.Scatter(
    x=fraudulentos['amount'],
    y=fraudulentos['anomaly_score'],
    mode='markers',
    name='ALERTA: FRAUDE REAL',
    marker=dict(
        color='red',
        size=12,      # Más grandes
        opacity=1.0,  # Sólidas
        symbol='x',   # Forma distintiva
        line=dict(width=2, color='darkred')
    )
))

# 3. Diseño y Leyenda
fig.update_layout(
    title="Detección de Anomalías: Error de Reconstrucción (Autoencoder)",
    xaxis_title="Monto de la Transacción",
    yaxis_title="Score de Anomalía (Error)",
    template="plotly_dark",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(0,0,0,0.5)" # Fondo de leyenda semi-transparente
    )
)

# Renderizado
fig.show(renderer="notebook_connected")

In [26]:
import plotly.figure_factory as ff

# Preparamos los datos
error_normal = normales['anomaly_score'].tolist()
error_fraude = fraudulentos['anomaly_score'].tolist()

# Creamos un gráfico de densidad (Distplot)
fig_dist = ff.create_distplot(
    [error_normal, error_fraude], 
    ['Normal', 'Fraude'], 
    bin_size=.05, 
    curve_type='kde', # Curva de densidad suave
    colors=['#3498db', '#e74c3c']
)

fig_dist.update_layout(
    title="¿Funciona el AE? Separación de Errores entre Clases",
    xaxis_title="Score de Anomalía (Error de Reconstrucción)",
    template="plotly_dark"
)
fig_dist.show(renderer="notebook_connected")

In [27]:
# Definimos el umbral (Threshold)
threshold = normales['anomaly_score'].quantile(0.95)
print(f"Umbral definido (Percentil 95 de normales): {threshold:.4f}")

# Clasificamos como anomalía si supera el umbral
sample_pd['prediccion_anomalia'] = (sample_pd['anomaly_score'] > threshold).astype(int)

# Creamos una matriz de confusión simple con Pandas
matriz = pd.crosstab(
    sample_pd['label_real'], 
    sample_pd['prediccion_anomalia'], 
    rownames=['Realidad'], 
    colnames=['Predicción AE']
)

print("\n--- Matriz de Confusión ---")
print(matriz)

# Visualización rápida con Plotly
fig_heatmap = px.imshow(
    matriz, 
    text_auto=True, 
    color_continuous_scale='YlOrRd',
    title="Matriz de Confusión: Detección por Umbral"
)
fig_heatmap.show(renderer="notebook_connected")

Umbral definido (Percentil 95 de normales): 0.0257

--- Matriz de Confusión ---
Predicción AE     0    1
Realidad                
0.0            9394  434
1.0             231   52


In [28]:
from sklearn.metrics import precision_recall_curve, auc

precision, recall, _ = precision_recall_curve(sample_pd['label_real'], sample_pd['anomaly_score'])
area_bajo_curva = auc(recall, precision)

fig_pr = px.area(
    x=recall, y=precision,
    title=f"Curva Precision-Recall (AUC: {area_bajo_curva:.4f})",
    labels=dict(x='Recall (Sensibilidad)', y='Precision (Precisión)'),
    template="plotly_dark"
)
fig_pr.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0.5, y1=0.5) # Línea base
fig_pr.show(renderer="notebook_connected")

In [ ]:
spark.stop()